In [ ]:
import pandas as pd
import numpy as np
import math
import os

DATA_DIR = "./data"
OUTPUT_DIR = "./outputs"

np.random.seed(42)

In [ ]:
df = pd.read_excel(os.path.join(OUTPUT_DIR, 'dataset_with_sentiment.xlsx'))

In [3]:
print(f"Full dataset size: {len(df)}")

Full dataset size: 5715


In [4]:
# --- Determine sample size using Slovin's formula (e = 0.05, 95% confidence) ---
N = len(df)
e = 0.05
SAMPLE_SIZE = math.ceil(N / (1 + N * e**2))

print(f"Population size (N): {N}")
print(f"Margin of error (e): {e}")
print(f"Minimum required sample size (Slovin's formula): {SAMPLE_SIZE}")

annotation_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"\nTotal annotation sample size: {len(annotation_sample)}")
print(f"\nExpected class composition (based on existing IndoBERT labels, "
      f"shown here only to preview expected balance - NOT used for sampling):")
print(annotation_sample['sentiment'].value_counts())

Population size (N): 5715
Margin of error (e): 0.05
Minimum required sample size (Slovin's formula): 374

Total annotation sample size: 374

Expected class composition (based on existing IndoBERT labels, shown here only to preview expected balance - NOT used for sampling):
sentiment
neutral     249
negative     66
positive     59
Name: count, dtype: int64


In [5]:
##Create blind annotation file (no model predictions visible to annotators)
annotation_blind = annotation_sample[['tweet_id', 'tweet', 'tweet_clean']].copy()
annotation_blind['annotator_1_label'] = ''
annotation_blind['annotator_2_label'] = ''
annotation_blind['final_label'] = ''

In [ ]:
# Shuffle row order again for good measure (sample() already shuffles, 
# this is just an extra safety step)
annotation_blind = annotation_blind.sample(frac=1, random_state=42).reset_index(drop=True)
 
annotation_blind.to_excel(os.path.join(OUTPUT_DIR, 'annotation_sample_blind_random.xlsx'), index=False)

## Manual Annotation Step (offline, outside this notebook)

At this point, `annotation_sample_blind_random.xlsx` was distributed to two independent annotators, 
who labelled each tweet as `positive`, `negative`, or `neutral` without seeing model predictions or 
each other's labels. The completed file (with `annotator_1_label` and `annotator_2_label` filled in) 
is provided in this repository as `outputs/annotation_sample_final.xlsx` (tweet text removed; 
`tweet_id` can be used to re-link to tweet content where permitted).

Since this step involves human judgment, it cannot be re-executed by running code — the completed 
annotation file is provided directly so that downstream analysis (inter-annotator agreement, 
model benchmarking) can be fully reproduced from this point forward.

In [7]:
# Keep a separate reference file with model predictions for later evaluation
# (NOT shown to annotators during the labelling process)
reference = annotation_sample[
    ['tweet_id', 'tweet_clean', 'sentiment_roberta', 'sentiment_model2', 'sentiment']
].copy()
reference = reference.loc[annotation_blind.index] if False else reference  # keep order aligned via tweet_id merge below

In [ ]:
# Re-align reference to match the shuffled annotation_blind order via tweet_id
reference = annotation_blind[['tweet_id']].merge(reference, on='tweet_id', how='left')
reference.to_excel(os.path.join(OUTPUT_DIR, 'reference_model_predictions_random.xlsx'), index=False)

Checking

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

annotated = pd.read_excel(os.path.join(OUTPUT_DIR, 'annotation_sample_blind_random.xlsx'))  # now filled in

# Quick sanity check before computing anything
print(annotated[['annotator_1_label', 'annotator_2_label']].isna().sum())
print(annotated['annotator_1_label'].unique())
print(annotated['annotator_2_label'].unique())

annotator_1_label    0
annotator_2_label    0
dtype: int64
<StringArray>
['Neutral', 'Negative', 'Positive']
Length: 3, dtype: str
<StringArray>
['Neutral', 'Negative', 'Positive']
Length: 3, dtype: str


In [13]:
missing_a1 = annotated[annotated['annotator_1_label'].isna()]
missing_a2 = annotated[annotated['annotator_2_label'].isna()]

print("Missing in annotator_1_label:")
print(missing_a1[['tweet_id', 'tweet_clean']])
print()
print("Missing in annotator_2_label:")
print(missing_a2[['tweet_id', 'tweet_clean']])

Missing in annotator_1_label:
Empty DataFrame
Columns: [tweet_id, tweet_clean]
Index: []

Missing in annotator_2_label:
Empty DataFrame
Columns: [tweet_id, tweet_clean]
Index: []


In [14]:
kappa = cohen_kappa_score(annotated['annotator_1_label'], annotated['annotator_2_label'])
print(f"Cohen's Kappa: {kappa:.4f}")

if kappa < 0.20:
    interp = "slight"
elif kappa < 0.40:
    interp = "fair"
elif kappa < 0.60:
    interp = "moderate"
elif kappa < 0.80:
    interp = "substantial"
else:
    interp = "almost perfect"
print(f"Agreement level ({interp})")

agree_pct = (annotated['annotator_1_label'] == annotated['annotator_2_label']).mean() * 100
print(f"Raw agreement: {agree_pct:.1f}%")

Cohen's Kappa: 0.5280
Agreement level (moderate)
Raw agreement: 78.6%


In [ ]:
disagreements = annotated[annotated['annotator_1_label'] != annotated['annotator_2_label']]
print(f"Disagreements: {len(disagreements)} of {len(annotated)} ({100*len(disagreements)/len(annotated):.1f}%)")
disagreements.to_excel(os.path.join(OUTPUT_DIR, 'disagreements_to_resolve.xlsx'), index=False)

Disagreements: 80 of 374 (21.4%)


## Manual Disagreement Resolution (offline, outside this notebook)

The 80 tweets where annotators disagreed were reviewed by a third pass to assign a final label. 
The resolved file is provided as `outputs/disagreements_to_resolve.xlsx` with the `resolved_label` 
column completed.

In [ ]:
import pandas as pd

annotated = pd.read_excel(os.path.join(OUTPUT_DIR, 'annotation_sample_blind_random.xlsx'))
resolved = pd.read_excel(os.path.join(OUTPUT_DIR, 'disagreements_to_resolve.xlsx'))

# Clean casing on all label columns
annotated['annotator_1_label'] = annotated['annotator_1_label'].str.lower().str.strip()
annotated['annotator_2_label'] = annotated['annotator_2_label'].str.lower().str.strip()
resolved['resolved_label'] = resolved['resolved_label'].str.lower().str.strip()

# Step 1: default final_label to annotator_1_label
# (this is correct for all AGREEMENT rows, since annotator_1 == annotator_2 there)
annotated['final_label'] = annotated['annotator_1_label']

# Step 2: overwrite with resolved_label wherever there was a disagreement
resolved_map = resolved.set_index('tweet_id')['resolved_label']
annotated['final_label'] = annotated.apply(
    lambda row: resolved_map[row['tweet_id']] if row['tweet_id'] in resolved_map.index else row['final_label'],
    axis=1
)

# --- Sanity checks before saving ---
print("Total rows:", len(annotated))
print("Missing final_label:", annotated['final_label'].isna().sum())
print("Unique final_label values:", annotated['final_label'].unique())
print()
print("Final label distribution:")
print(annotated['final_label'].value_counts())
print()

# Verify: for agreement rows, final_label should equal annotator_1_label (and annotator_2_label)
agree_rows = annotated[annotated['annotator_1_label'] == annotated['annotator_2_label']]
check = (agree_rows['final_label'] == agree_rows['annotator_1_label']).all()
print("Agreement rows correctly preserved:", check)

# Verify: for disagreement rows, final_label should match resolved_label
disagree_rows = annotated[annotated['annotator_1_label'] != annotated['annotator_2_label']]
check2 = (disagree_rows['final_label'] == disagree_rows['tweet_id'].map(resolved_map)).all()
print("Disagreement rows correctly resolved:", check2)

annotated.to_excel(os.path.join(OUTPUT_DIR, 'annotation_sample_final.xlsx'), index=False)
print("\nSaved to annotation_sample_final.xlsx")

Total rows: 374
Missing final_label: 0
Unique final_label values: <StringArray>
['neutral', 'negative', 'positive']
Length: 3, dtype: str

Final label distribution:
final_label
neutral     286
positive     45
negative     43
Name: count, dtype: int64

Agreement rows correctly preserved: True
Disagreement rows correctly resolved: True

Saved to annotation_sample_final.xlsx
